# Per-Bin Decay Rate Optimization

Uses Brent's method (derivative-free scalar optimization) to find the optimal
time-decay rate for each of the top bins that cover 90%+ of scored jobs.

**Method:** For each bin, minimize MAE as a function of decay rate using
`scipy.optimize.minimize_scalar` with bounds [0, 0.15].

**Dataset:** NLR Kestrel, expanded window (~193 days, 2.7M rows)
**Model:** XGBoost Adjusted (200 trees, depth 12)
**Preprocessing:** SVD=64, OHE=512
**Lookback:** 120 days, 120 windows × 6h
**Bins optimized:** Top 8 bins covering ~92% of scored jobs

## 1. Setup

In [1]:
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from scipy.optimize import minimize_scalar

from hpc_oda_commons.models.experimental.xgboost_adjusted_model import (
    ExperimentalXGBoostAdjustedConfig, ExperimentalXGBoostAdjustedModel,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

REPO_ROOT = Path.cwd().parent.parent
DATA_PATH = REPO_ROOT / 'workspace' / 'data' / 'datasets' / 'nlr_kestrel' / 'data.parquet'
from tqdm.notebook import tqdm


In [2]:
# Load expanded window
table = pq.read_table(DATA_PATH)
lo = datetime(2025, 1, 1, tzinfo=timezone.utc)
hi = datetime(2025, 6, 26, tzinfo=timezone.utc) + timedelta(days=1)
sc = table.column('submit_time')
ec = table.column('end_time')
mask = pc.and_(
    pc.less(sc, pa.scalar(hi, type=sc.type)),
    pc.greater_equal(ec, pa.scalar(lo, type=ec.type)),
)
df = table.filter(mask).to_pandas()
rows_all = df.to_dict('records')
print(f'Loaded: {len(rows_all):,} rows')
print(f'Span: {(df["submit_time"].max() - df["submit_time"].min()).days} days')

Loaded: 2,702,850 rows
Span: 193 days


## 2. Configuration

In [3]:
N_WINDOWS = 120
TEST_WINDOW_HOURS = 6
TRAINING_LOOKBACK_DAYS = 120

# Optimization bounds
DECAY_LOWER = 0.0
DECAY_UPPER = 0.15
DECAY_TOLERANCE = 0.005  # stop when rate is precise to ±0.005

POWER_USER_PERCENTILE = 0.99

BIN_EDGES_H = [0, 2, 4, 24, 48, float('inf')]
BIN_LABELS = ['<=2h', '2-4h', '4-24h', '24-48h', '>48h']

def make_config(decay_rate):
    return ExperimentalXGBoostAdjustedConfig(
        n_windows=N_WINDOWS,
        test_window_hours=TEST_WINDOW_HOURS,
        training_lookback_days=TRAINING_LOOKBACK_DAYS,
        max_svd_components=64,
        target_max_one_hot_width=512,
        random_state=42,
        n_estimators=200,
        max_depth=12,
        learning_rate=0.03,
        min_child_weight=5,
        gamma=0.1,
        time_decay_rate=decay_rate,
        estimator_n_jobs=12,
    )

print(f'Optimization bounds: [{DECAY_LOWER}, {DECAY_UPPER}]')
print(f'Tolerance: {DECAY_TOLERANCE}')
print(f'At upper bound (rate={DECAY_UPPER}): weight at 120 days = {np.exp(-DECAY_UPPER*120):.6f}')

Optimization bounds: [0.0, 0.15]
Tolerance: 0.005
At upper bound (rate=0.15): weight at 120 days = 0.000000


## 3. Identify top bins (covering 90%+ of scored jobs)

In [4]:
def assign_bin(row):
    wc_h = (row.get('requested_seconds') or 0) / 3600
    for i in range(len(BIN_EDGES_H) - 1):
        if wc_h <= BIN_EDGES_H[i + 1]:
            return BIN_LABELS[i]
    return BIN_LABELS[-1]

# Identify power users
user_counts = Counter(r.get('user') for r in rows_all)
threshold = np.percentile(list(user_counts.values()), POWER_USER_PERCENTILE * 100)
power_users = {u for u, c in user_counts.items() if c >= threshold}

# Build all bins
all_bins = {}
for row in rows_all:
    user = row.get('user')
    bl = assign_bin(row)
    if user in power_users:
        key = f'power {user[:7]}/{bl}'
    else:
        key = f'non-power/{bl}'
    all_bins.setdefault(key, []).append(row)

# Sort by size and find top bins covering 90%
valid_bins = {k: v for k, v in all_bins.items() if len(v) >= 100}
sorted_bins = sorted(valid_bins.items(), key=lambda x: -len(x[1]))
total_rows = sum(len(v) for v in valid_bins.values())

cumulative = 0
top_bins = []
for name, rows in sorted_bins:
    cumulative += len(rows)
    top_bins.append((name, rows))
    if cumulative / total_rows >= 0.90:
        break

print(f'Power users: {len(power_users)}')
print(f'Total valid bins: {len(valid_bins)}')
print(f'Top bins for 90% coverage: {len(top_bins)}')
print(f'Coverage: {cumulative/total_rows*100:.1f}%')
print(f'\n{"Bin":<35} {"Rows":>10}')
print('-' * 48)
for name, rows in top_bins:
    print(f'{name:<35} {len(rows):>10,}')

Power users: 8
Total valid bins: 32
Top bins for 90% coverage: 11
Coverage: 91.1%

Bin                                       Rows
------------------------------------------------
power c6b628f/<=2h                     803,417
non-power/4-24h                        390,763
non-power/<=2h                         361,292
non-power/24-48h                       263,179
non-power/2-4h                         192,632
power f781187/2-4h                     128,162
power b5fe0d4/24-48h                    78,739
power 24b498e/4-24h                     74,336
power cdb4f00/4-24h                     60,319
power 90119ab/4-24h                     59,374
power 05630e6/4-24h                     49,826


## 4. Optimize decay rate per bin

For each bin, use Brent's method to find the decay rate that minimizes MAE.
The optimizer adaptively chooses ~5-8 evaluation points per bin.

In [ ]:
import sys, json as _json
from tqdm.notebook import tqdm as tqdm_notebook

PROGRESS_FILE = REPO_ROOT / 'workspace' / 'decay_optimization_progress.json'

def _save_progress(results, status='running'):
    """Write progress to disk so it can be checked from terminal."""
    serializable = {}
    for k, v in results.items():
        serializable[k] = {sk: (float(sv) if isinstance(sv, (int, float)) else sv) for sk, sv in v.items()}
    PROGRESS_FILE.write_text(_json.dumps({
        'status': status,
        'bins_completed': len(results),
        'results': serializable,
    }, indent=2))

optimization_results = {}
total_start = time.time()

bin_pbar = tqdm_notebook(top_bins, desc='Bins', unit='bin')
for bin_idx, (bin_name, bin_rows) in enumerate(bin_pbar):
    bin_pbar.set_postfix_str(f'{bin_name} ({len(bin_rows):,} rows)')

    eval_count = [0]
    bin_start = time.time()

    # tqdm bar for evaluations within this bin (Brent typically does 6-12)
    eval_pbar = tqdm_notebook(total=12, desc=f'  Evals [{bin_name}]', unit='eval', leave=False)

    def objective(rate):
        eval_count[0] += 1
        eval_start = time.time()
        try:
            config = make_config(rate)
            model = ExperimentalXGBoostAdjustedModel(config)
            payload = model.evaluate(bin_rows, verbose=False)
            mae = payload['mae']
            eval_time = (time.time() - eval_start) / 60
            eval_pbar.update(1)
            eval_pbar.set_postfix_str(f'rate={rate:.4f} MAE={mae:,.0f}s ({eval_time:.1f}min)')
            # Also save each eval to disk for terminal monitoring
            _save_progress({**optimization_results, f'{bin_name} (in progress)': {
                'eval': eval_count[0], 'last_rate': rate, 'last_mae': mae,
                'elapsed_min': (time.time() - bin_start) / 60, 'rows': len(bin_rows),
            }})
            return mae
        except Exception as e:
            eval_pbar.update(1)
            eval_pbar.set_postfix_str(f'rate={rate:.4f} FAILED')
            return 1e9

    # Run optimization
    result = minimize_scalar(
        objective,
        bounds=(DECAY_LOWER, DECAY_UPPER),
        method='bounded',
        options={'xatol': DECAY_TOLERANCE}
    )

    # Also evaluate flat (rate=0) for comparison
    flat_mae = objective(0.0) if eval_count[0] > 0 else None

    eval_pbar.close()
    bin_elapsed = (time.time() - bin_start) / 60

    optimization_results[bin_name] = {
        'optimal_rate': result.x,
        'optimal_mae': result.fun,
        'flat_mae': flat_mae,
        'evaluations': eval_count[0],
        'time_min': bin_elapsed,
        'rows': len(bin_rows),
    }

    improvement = (result.fun - flat_mae) / flat_mae * 100 if flat_mae else 0
    elapsed_total = (time.time() - total_start) / 60
    bin_pbar.set_postfix_str(
        f'{bin_name}: rate={result.x:.4f}, {improvement:+.1f}% ({bin_elapsed:.0f}min)'
    )

    _save_progress(optimization_results)

bin_pbar.close()
total_elapsed = (time.time() - total_start) / 60
print(f'\nOPTIMIZATION COMPLETE — {total_elapsed:.0f} minutes total', flush=True)
_save_progress(optimization_results, status='complete')

Bins:   0%|          | 0/11 [00:00<?, ?bin/s]

  Evals [power c6b628f/<=2h]:   0%|          | 0/12 [00:00<?, ?eval/s]

  Evals [non-power/4-24h]:   0%|          | 0/12 [00:00<?, ?eval/s]

## 5. Results

In [ ]:
print('=' * 80)
print('OPTIMAL DECAY RATE PER BIN')
print('=' * 80)

print(f'\n{"Bin":<35} {"Rows":>8} {"Optimal Rate":>12} {"Optimal MAE":>12} {"Flat MAE":>10} {"Improvement":>12}')
print('-' * 92)
for name, r in sorted(optimization_results.items(), key=lambda x: -x[1]['rows']):
    imp = (r['optimal_mae'] - r['flat_mae']) / r['flat_mae'] * 100 if r['flat_mae'] else 0
    print(f'{name:<35} {r["rows"]:>8,} {r["optimal_rate"]:>12.4f} {r["optimal_mae"]:>12,.0f}s {r["flat_mae"]:>10,.0f}s {imp:>+11.1f}%')

# Aggregate: per-bin-optimal vs flat everywhere
total_opt_weighted = sum(r['optimal_mae'] * r['rows'] for r in optimization_results.values())
total_flat_weighted = sum(r['flat_mae'] * r['rows'] for r in optimization_results.values() if r['flat_mae'])
total_rows = sum(r['rows'] for r in optimization_results.values())

agg_opt = total_opt_weighted / total_rows
agg_flat = total_flat_weighted / total_rows
agg_imp = (agg_opt - agg_flat) / agg_flat * 100

print(f'\n{"WEIGHTED AGGREGATE":<35} {total_rows:>8,} {"(per-bin)":>12} {agg_opt:>12,.0f}s {agg_flat:>10,.0f}s {agg_imp:>+11.1f}%')

In [ ]:
# Visualization: optimal rate per bin
names = sorted(optimization_results.keys(), key=lambda x: -optimization_results[x]['rows'])
rates = [optimization_results[n]['optimal_rate'] for n in names]
colors = ['seagreen' if 'power' in n else 'steelblue' for n in names]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(names[::-1], rates[::-1], color=colors[::-1])
ax.set_xlabel('Optimal Decay Rate')
ax.set_title('Optimal Time-Decay Rate Per Bin\n(Brent optimization, bounds [0, 0.15])')

for bar, rate in zip(bars, rates[::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{rate:.3f}', va='center', fontsize=9)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='seagreen', label='Power user bins'),
    Patch(color='steelblue', label='Non-power user bins'),
])
plt.tight_layout()
plt.show()

## 6. Pattern Analysis

In [ ]:
print('PATTERN ANALYSIS')
print('=' * 60)

# Do short-job bins prefer higher decay?
print('\nBy wallclock cluster:')
for bl in BIN_LABELS:
    matching = [(n, r) for n, r in optimization_results.items() if bl in n]
    if matching:
        rates = [r['optimal_rate'] for _, r in matching]
        print(f'  {bl}: mean optimal rate = {np.mean(rates):.4f} (n={len(rates)})')

# Power vs non-power
print('\nPower vs non-power users:')
power_rates = [r['optimal_rate'] for n, r in optimization_results.items() if 'power' in n and 'non-power' not in n]
np_rates = [r['optimal_rate'] for n, r in optimization_results.items() if 'non-power' in n]
if power_rates:
    print(f'  Power user bins: mean optimal rate = {np.mean(power_rates):.4f} (n={len(power_rates)})')
if np_rates:
    print(f'  Non-power bins:  mean optimal rate = {np.mean(np_rates):.4f} (n={len(np_rates)})')

# Overall
all_rates = [r['optimal_rate'] for r in optimization_results.values()]
print(f'\nOverall: mean={np.mean(all_rates):.4f}, std={np.std(all_rates):.4f}')
print(f'Range: [{min(all_rates):.4f}, {max(all_rates):.4f}]')

if np.std(all_rates) < 0.01:
    print('\nConclusion: all bins prefer roughly the same decay rate.')
    print('A single global rate would work just as well as per-bin optimization.')
else:
    print('\nConclusion: bins prefer different decay rates.')
    print('Per-bin optimization provides value over a single global rate.')